# Focusboard Secure Task Manager

This notebook bundles the complete Flask project source and CI configuration. Run the cells from top to bottom. The app is a local single-workspace task manager; it has no login and must not be exposed to the public internet.

The app stores user-created tasks in SQLite. It does not use an external dataset.

In [ ]:
from pathlib import Path

for folder in ["templates", "static", ".github/workflows", "scripts"]:
    Path(folder).mkdir(parents=True, exist_ok=True)
print("Project folders are ready.")

In [ ]:
%%writefile requirements.txt
Flask==3.1.2
gunicorn==23.0.0
blinker==1.9.0
click==8.5.0
itsdangerous==2.2.0
Jinja2==3.1.6
MarkupSafe==3.0.4
packaging==26.3
Werkzeug==3.1.9


In [ ]:
%%writefile app.py
"""Secure Task Manager: a small Flask UI and JSON API backed by SQLite."""

from __future__ import annotations

import os
import sqlite3
import tempfile
from contextlib import closing, contextmanager
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Any

from flask import Flask, jsonify, render_template, request


def create_app(database_path: str | Path | None = None) -> Flask:
    app = Flask(__name__)
    default_path = Path(tempfile.gettempdir()) / "secure-task-manager.db"
    configured_path = database_path or os.environ.get("DATABASE_PATH", str(default_path))
    app.config["DATABASE_PATH"] = str(configured_path)

    def connect_db() -> sqlite3.Connection:
        connection = sqlite3.connect(app.config["DATABASE_PATH"], timeout=10)
        connection.row_factory = sqlite3.Row
        connection.execute("PRAGMA foreign_keys = ON")
        return connection

    @contextmanager
    def db_session():
        with closing(connect_db()) as connection:
            with connection:
                yield connection

    def initialize_db() -> None:
        Path(app.config["DATABASE_PATH"]).parent.mkdir(parents=True, exist_ok=True)
        with db_session() as db:
            db.execute("PRAGMA journal_mode = WAL")
            db.execute(
                """CREATE TABLE IF NOT EXISTS tasks (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    title TEXT NOT NULL CHECK(length(title) BETWEEN 1 AND 120),
                    description TEXT NOT NULL DEFAULT '' CHECK(length(description) <= 2000),
                    status TEXT NOT NULL DEFAULT 'open' CHECK(status IN ('open', 'in_progress', 'done')),
                    priority TEXT NOT NULL DEFAULT 'medium' CHECK(priority IN ('low', 'medium', 'high')),
                    due_date TEXT,
                    created_at TEXT NOT NULL,
                    updated_at TEXT NOT NULL
                )"""
            )

    def task_dict(row: sqlite3.Row) -> dict[str, Any]:
        return dict(row)

    def read_payload() -> tuple[dict[str, Any] | None, tuple[Any, int] | None]:
        payload = request.get_json(silent=True)
        if not isinstance(payload, dict):
            return None, (jsonify(error="Send a JSON object in the request body."), 400)
        return payload, None

    def validate_task(payload: dict[str, Any], partial: bool = False) -> tuple[dict[str, Any] | None, str | None]:
        allowed = {"title", "description", "status", "priority", "due_date"}
        unexpected = set(payload) - allowed
        if unexpected:
            return None, "Unsupported field(s): " + ", ".join(sorted(unexpected))

        values: dict[str, Any] = {}
        if "title" in payload or not partial:
            title = payload.get("title", "")
            if not isinstance(title, str) or not title.strip() or len(title.strip()) > 120:
                return None, "Title is required and must be 1–120 characters."
            values["title"] = title.strip()

        if "description" in payload or not partial:
            description = payload.get("description", "")
            if not isinstance(description, str) or len(description) > 2000:
                return None, "Description must be 2,000 characters or fewer."
            values["description"] = description.strip()

        if "status" in payload or not partial:
            status = payload.get("status", "open")
            if not isinstance(status, str) or status not in {"open", "in_progress", "done"}:
                return None, "Status must be open, in_progress, or done."
            values["status"] = status

        if "priority" in payload or not partial:
            priority = payload.get("priority", "medium")
            if not isinstance(priority, str) or priority not in {"low", "medium", "high"}:
                return None, "Priority must be low, medium, or high."
            values["priority"] = priority

        if "due_date" in payload or not partial:
            due_date = payload.get("due_date") or None
            if due_date is not None:
                if not isinstance(due_date, str):
                    return None, "Due date must use YYYY-MM-DD format."
                try:
                    date.fromisoformat(due_date)
                except ValueError:
                    return None, "Due date must use YYYY-MM-DD format."
            values["due_date"] = due_date

        return values, None

    initialize_db()

    @app.after_request
    def add_security_headers(response):
        response.headers.setdefault("X-Content-Type-Options", "nosniff")
        response.headers.setdefault("X-Frame-Options", "DENY")
        response.headers.setdefault("Referrer-Policy", "no-referrer")
        response.headers.setdefault("Content-Security-Policy", "default-src 'self'; style-src 'self'; script-src 'self'; connect-src 'self'; img-src 'self' data:; base-uri 'self'; frame-ancestors 'none'")
        return response

    @app.get("/")
    def index():
        return render_template("index.html")

    @app.get("/health")
    def health():
        try:
            with db_session() as db:
                db.execute("SELECT 1")
        except sqlite3.Error:
            app.logger.exception("Database health check failed")
            return jsonify(status="error"), 503
        return jsonify(status="ok", service="secure-task-manager")

    @app.get("/api/tasks")
    def list_tasks():
        status = request.args.get("status")
        query = request.args.get("q", "").strip()[:120]
        if status and status not in {"open", "in_progress", "done"}:
            return jsonify(error="Unknown status filter."), 400
        sql = "SELECT * FROM tasks WHERE 1 = 1"
        parameters: list[Any] = []
        if status:
            sql += " AND status = ?"
            parameters.append(status)
        if query:
            sql += " AND (title LIKE ? OR description LIKE ?)"
            parameters.extend((f"%{query}%", f"%{query}%"))
        sql += " ORDER BY CASE status WHEN 'open' THEN 0 WHEN 'in_progress' THEN 1 ELSE 2 END, " \
               "CASE priority WHEN 'high' THEN 0 WHEN 'medium' THEN 1 ELSE 2 END, updated_at DESC"
        with db_session() as db:
            rows = db.execute(sql, parameters).fetchall()
            counts = db.execute(
                "SELECT COUNT(*) AS total, SUM(status = 'open') AS open, "
                "SUM(status = 'in_progress') AS in_progress, SUM(status = 'done') AS done FROM tasks"
            ).fetchone()
        return jsonify(tasks=[task_dict(row) for row in rows], stats={key: counts[key] or 0 for key in counts.keys()})

    @app.post("/api/tasks")
    def create_task():
        payload, error = read_payload()
        if error:
            return error
        values, message = validate_task(payload)
        if message:
            return jsonify(error=message), 400
        now = datetime.now(timezone.utc).isoformat(timespec="seconds")
        with db_session() as db:
            cursor = db.execute(
                "INSERT INTO tasks(title, description, status, priority, due_date, created_at, updated_at) "
                "VALUES (?, ?, ?, ?, ?, ?, ?)",
                (values["title"], values["description"], values["status"], values["priority"], values["due_date"], now, now),
            )
            row = db.execute("SELECT * FROM tasks WHERE id = ?", (cursor.lastrowid,)).fetchone()
        return jsonify(task=task_dict(row)), 201

    @app.patch("/api/tasks/<int:task_id>")
    def update_task(task_id: int):
        payload, error = read_payload()
        if error:
            return error
        if not payload:
            return jsonify(error="Provide at least one field to update."), 400
        values, message = validate_task(payload, partial=True)
        if message:
            return jsonify(error=message), 400
        values["updated_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
        assignments = ", ".join(f"{field} = ?" for field in values)
        with db_session() as db:
            cursor = db.execute(f"UPDATE tasks SET {assignments} WHERE id = ?", [*values.values(), task_id])
            if cursor.rowcount == 0:
                return jsonify(error="Task not found."), 404
            row = db.execute("SELECT * FROM tasks WHERE id = ?", (task_id,)).fetchone()
        return jsonify(task=task_dict(row))

    @app.delete("/api/tasks/<int:task_id>")
    def delete_task(task_id: int):
        with db_session() as db:
            cursor = db.execute("DELETE FROM tasks WHERE id = ?", (task_id,))
        if cursor.rowcount == 0:
            return jsonify(error="Task not found."), 404
        return "", 204

    @app.errorhandler(404)
    def not_found(_error):
        if request.path.startswith("/api/"):
            return jsonify(error="Not found."), 404
        return render_template("index.html"), 404

    @app.errorhandler(413)
    def payload_too_large(_error):
        return jsonify(error="Request body is too large."), 413

    app.config["MAX_CONTENT_LENGTH"] = 64 * 1024
    return app


app = create_app()

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=int(os.environ.get("PORT", "5000")))


In [ ]:
%%writefile templates/index.html
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <meta name="theme-color" content="#101a2e">
  <meta name="description" content="A focused task manager with a secure Flask API and automated CI security gates.">
  <title>Focusboard — Secure Task Manager</title>
  <link rel="stylesheet" href="{{ url_for('static', filename='app.css') }}">
  <script src="{{ url_for('static', filename='app.js') }}" defer></script>
</head>
<body>
  <div class="app-shell">
    <aside class="sidebar" aria-label="Main navigation">
      <a class="brand" href="/" aria-label="Focusboard home">
        <span class="brand-mark" aria-hidden="true">F</span>
        <span>focusboard<span class="brand-period">.</span></span>
      </a>
      <div class="workspace-label">WORKSPACE</div>
      <nav class="nav-list" aria-label="Task views">
        <button class="nav-item active" data-filter="all"><span class="nav-icon">▦</span><span>All tasks</span><span class="nav-count" id="nav-total">0</span></button>
        <button class="nav-item" data-filter="open"><span class="nav-icon">◷</span><span>To do</span><span class="nav-count" id="nav-open">0</span></button>
        <button class="nav-item" data-filter="in_progress"><span class="nav-icon">↗</span><span>In progress</span><span class="nav-count" id="nav-progress">0</span></button>
        <button class="nav-item" data-filter="done"><span class="nav-icon">✓</span><span>Completed</span><span class="nav-count" id="nav-done">0</span></button>
      </nav>
      <div class="sidebar-bottom">
        <div class="security-note"><span class="security-dot"></span><div><strong>Secure by design</strong><small>Input validation · SQL parameters</small></div></div>
        <div class="sidebar-foot">FOCUSBOARD <span>·</span> PERSONAL WORKSPACE</div>
      </div>
    </aside>

    <main class="main-content">
      <header class="topbar">
        <div class="breadcrumb">Workspace <span>/</span> <strong id="view-name">All tasks</strong></div>
        <div class="top-actions"><span class="local-badge"><span></span> Local workspace</span><span class="avatar" aria-label="Personal workspace">P</span></div>
      </header>

      <section class="page-heading">
        <div>
          <p class="eyebrow" id="today-label">YOUR DAY, IN FOCUS</p>
          <h1 id="heading-title">All tasks<span class="heading-dot">.</span></h1>
          <p class="subtitle">A clear view of what needs your attention.</p>
        </div>
        <button class="primary-button" id="add-task"><span aria-hidden="true">＋</span> New task</button>
      </section>

      <section class="stats-grid" aria-label="Task overview">
        <article class="stat-card"><div class="stat-top"><span>Total tasks</span><span class="stat-icon indigo">▦</span></div><strong id="stat-total">0</strong><small>Across your workspace</small></article>
        <article class="stat-card"><div class="stat-top"><span>To do</span><span class="stat-icon amber">◷</span></div><strong id="stat-open">0</strong><small>Ready when you are</small></article>
        <article class="stat-card"><div class="stat-top"><span>In progress</span><span class="stat-icon blue">↗</span></div><strong id="stat-progress">0</strong><small>Currently underway</small></article>
        <article class="stat-card"><div class="stat-top"><span>Completed</span><span class="stat-icon green">✓</span></div><strong id="stat-done">0</strong><small>One step at a time</small></article>
      </section>

      <section class="task-panel" aria-labelledby="task-section-title">
        <div class="panel-header">
          <div><h2 id="task-section-title">Your tasks</h2><p id="task-count">Loading your workspace…</p></div>
          <label class="search-box"><span aria-hidden="true">⌕</span><input id="search" type="search" maxlength="120" placeholder="Search tasks" aria-label="Search tasks"></label>
        </div>
        <div class="list-head" aria-hidden="true"><span>TASK</span><span>PRIORITY</span><span>STATUS</span><span>DUE DATE</span><span></span></div>
        <div id="task-list" class="task-list" aria-live="polite"></div>
        <div id="empty-state" class="empty-state" hidden>
          <div class="empty-art" aria-hidden="true"><span>✓</span></div>
          <h3 id="empty-title">A little room to think.</h3>
          <p id="empty-copy">Your task list is clear. Add a task to get started.</p>
          <button class="text-button" id="empty-add">Create your first task <span>→</span></button>
        </div>
      </section>
      <footer class="page-footer"><span>FOCUSBOARD</span><span>Thoughtful work, one task at a time.</span><a href="/health">System status <i></i></a></footer>
    </main>
  </div>

  <dialog id="task-dialog" class="task-dialog" aria-labelledby="dialog-title">
    <form id="task-form">
      <div class="dialog-heading"><div><p class="eyebrow">TASK DETAILS</p><h2 id="dialog-title">Create a task</h2></div><button type="button" class="icon-button" id="close-dialog" aria-label="Close">×</button></div>
      <label class="field-label" for="title">Title <span>*</span></label>
      <input id="title" name="title" required maxlength="120" placeholder="What needs to get done?">
      <label class="field-label" for="description">Description <small>Optional</small></label>
      <textarea id="description" name="description" rows="4" maxlength="2000" placeholder="Add a little context…"></textarea>
      <div class="field-row">
        <div><label class="field-label" for="priority">Priority</label><select id="priority" name="priority"><option value="low">Low</option><option value="medium" selected>Medium</option><option value="high">High</option></select></div>
        <div><label class="field-label" for="status">Status</label><select id="status" name="status"><option value="open">To do</option><option value="in_progress">In progress</option><option value="done">Completed</option></select></div>
      </div>
      <label class="field-label" for="due-date">Due date <small>Optional</small></label>
      <input id="due-date" name="due_date" type="date">
      <p class="form-error" id="form-error" role="alert" hidden></p>
      <div class="dialog-actions"><button type="button" class="secondary-button" id="cancel-dialog">Cancel</button><button class="primary-button" id="save-task">Save task</button></div>
    </form>
  </dialog>
  <div id="toast" class="toast" role="status" aria-live="polite"></div>
</body>
</html>


In [ ]:
%%writefile static/app.css
:root{font-family:Inter,ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;color:#182338;background:#f6f7fb;font-synthesis:none;text-rendering:optimizeLegibility;font-smooth:always;font-kerning:normal;--muted:#8993a6;--line:#e9ecf2;--navy:#111b31;--purple:#6259e8;--shadow:0 10px 28px rgba(28,39,66,.06)}
*{box-sizing:border-box}body{margin:0;min-width:360px;min-height:100vh}.app-shell{min-height:100vh}.sidebar{position:fixed;inset:0 auto 0 0;width:252px;background:var(--navy);color:#d9deea;padding:30px 18px 20px;display:flex;flex-direction:column;z-index:2}.brand{display:flex;align-items:center;gap:11px;color:#fff;text-decoration:none;font-weight:750;font-size:19px;letter-spacing:-.6px;padding:0 8px}.brand-mark{width:31px;height:31px;border-radius:10px;display:grid;place-items:center;background:linear-gradient(145deg,#8079ff,#594ce1);font-weight:800;font-size:17px;box-shadow:0 5px 14px #564bd65c}.brand-period{color:#8982ff}.workspace-label{font-size:10px;letter-spacing:1.4px;font-weight:750;color:#737f98;margin:48px 10px 14px}.nav-list{display:grid;gap:5px}.nav-item{border:0;background:none;color:#aab3c5;width:100%;display:flex;align-items:center;gap:12px;text-align:left;padding:11px 12px;border-radius:9px;font-family:inherit;font-size:13px;font-weight:500;cursor:pointer;transition:background .15s,color .15s}.nav-item:hover{background:#ffffff0d;color:#fff}.nav-item.active{background:#ffffff12;color:#fff;box-shadow:inset 2px 0 #8c84ff}.nav-icon{font-size:17px;width:17px;text-align:center;color:#8993ab}.nav-item.active .nav-icon{color:#a49eff}.nav-count{margin-left:auto;color:#818ba2;font-size:11px}.nav-item.active .nav-count{color:#c4c8d8}.sidebar-bottom{margin-top:auto}.security-note{display:flex;gap:10px;align-items:flex-start;background:#ffffff0a;border:1px solid #ffffff12;border-radius:10px;padding:13px 12px}.security-dot{flex:none;width:8px;height:8px;background:#50c995;border-radius:50%;margin-top:5px;box-shadow:0 0 0 4px #50c9951c}.security-note strong{display:block;font-size:11px;color:#e2e7f1;font-weight:650}.security-note small{display:block;color:#8490a9;font-size:9px;margin-top:4px}.sidebar-foot{font-size:8px;letter-spacing:1px;color:#69758c;margin:18px 3px 0}.sidebar-foot span{padding:0 4px;color:#8c84ff}.main-content{margin-left:252px;max-width:1600px;padding:0 48px 25px}.topbar{height:76px;border-bottom:1px solid #e9ecf1;display:flex;align-items:center;justify-content:space-between}.breadcrumb{font-size:12px;color:#8993a6}.breadcrumb span{padding:0 10px;color:#c3c8d2}.breadcrumb strong{color:#47536a;font-weight:600}.top-actions{display:flex;align-items:center;gap:17px}.local-badge{display:flex;align-items:center;gap:7px;border:1px solid #e6eaf0;background:#fff;padding:7px 10px;border-radius:20px;color:#717d92;font-size:10px}.local-badge span{width:6px;height:6px;background:#52c598;border-radius:50%}.avatar{display:grid;place-items:center;width:30px;height:30px;border-radius:50%;background:#e8e7ff;color:#5e54df;font-size:11px;font-weight:700}.page-heading{display:flex;align-items:flex-end;justify-content:space-between;padding:37px 0 25px}.eyebrow{font-size:9px;letter-spacing:1.65px;font-weight:750;color:#929bad;margin:0 0 9px}.page-heading h1{font-size:29px;letter-spacing:-1.1px;margin:0;color:#1c2941}.heading-dot{color:#655be8}.subtitle{font-size:12px;color:#8993a6;margin:8px 0 0}.primary-button,.secondary-button{display:inline-flex;align-items:center;justify-content:center;gap:8px;border:0;border-radius:8px;min-height:39px;padding:0 16px;font-family:inherit;font-size:11px;font-weight:600;cursor:pointer;transition:transform .15s,box-shadow .15s,background .15s}.primary-button{background:var(--purple);color:#fff;box-shadow:0 4px 10px #655be833}.primary-button:hover{background:#554bd6;box-shadow:0 6px 15px #655be844;transform:translateY(-1px)}.primary-button:disabled{opacity:.6;cursor:wait;transform:none}.primary-button>span{font-size:18px;font-weight:400}.secondary-button{background:#f1f2f6;color:#626d80}.secondary-button:hover{background:#e8eaf1}.stats-grid{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:14px;margin-bottom:22px}.stat-card{padding:16px 17px 15px;background:#fff;border:1px solid #eef0f5;border-radius:11px;box-shadow:0 3px 12px #27365406}.stat-top{display:flex;align-items:center;justify-content:space-between;color:#778297;font-size:10px;font-weight:550}.stat-icon{width:27px;height:27px;border-radius:8px;display:grid;place-items:center;font-size:14px}.indigo{color:#645be7;background:#efeeff}.amber{color:#db9c30;background:#fff5df}.blue{color:#3786d3;background:#eaf4ff}.green{color:#35a97e;background:#e9f8f1}.stat-card>strong{display:block;margin:10px 0 2px;font-size:24px;letter-spacing:-.8px;color:#202c42}.stat-card>small{color:#a0a8b7;font-size:9px}.task-panel{background:#fff;border:1px solid #edf0f5;border-radius:12px;box-shadow:0 3px 14px #27365407;min-height:390px}.panel-header{padding:19px 21px 15px;display:flex;align-items:center;justify-content:space-between}.panel-header h2{font-size:14px;letter-spacing:-.25px;margin:0;color:#243149}.panel-header p{font-size:10px;color:#9aa3b2;margin:5px 0 0}.search-box{width:205px;height:33px;border:1px solid #e9ecf2;border-radius:7px;display:flex;gap:8px;align-items:center;padding:0 10px;color:#9aa3b4}.search-box>span{font-size:18px;transform:rotate(-20deg)}.search-box input{min-width:0;width:100%;border:0;outline:0;background:transparent;color:#39445a;font-family:inherit;font-size:11px}.search-box input::placeholder{color:#a8afbc}.list-head{height:35px;background:#fafbfc;border-top:1px solid #f0f1f5;border-bottom:1px solid #f0f1f5;display:grid;grid-template-columns:minmax(200px,1fr) 120px 125px 110px 35px;align-items:center;padding:0 21px;gap:12px;font-size:8px;letter-spacing:1px;color:#9aa3b2;font-weight:700}.task-list{padding:0 21px}.task-row{min-height:67px;display:grid;grid-template-columns:minmax(200px,1fr) 120px 125px 110px 35px;align-items:center;gap:12px;border-bottom:1px solid #f0f1f5;animation:appear .25s ease both}.task-row:last-child{border-bottom:0}.task-main{display:flex;align-items:flex-start;gap:11px;min-width:0}.task-check{flex:none;margin-top:2px;width:16px;height:16px;border:1.5px solid #cfd5e0;border-radius:5px;background:#fff;color:transparent;cursor:pointer;display:grid;place-items:center;font-size:11px;padding:0}.task-check:hover{border-color:#665ce7}.task-check.is-done{background:#5dca9b;border-color:#5dca9b;color:#fff}.task-text{min-width:0}.task-title{font-size:11px;font-weight:600;color:#354158;line-height:1.45;word-break:break-word}.task-row.completed .task-title{text-decoration:line-through;color:#9ca5b3}.task-description{font-size:9px;color:#9aa3b2;margin-top:3px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;max-width:430px}.priority,.status-pill{display:inline-flex;width:max-content;align-items:center;gap:6px;border-radius:20px;padding:5px 8px;font-size:9px;font-weight:600}.priority::before{content:"";width:5px;height:5px;border-radius:50%}.priority-high{color:#d66e70;background:#fff0f0}.priority-high::before{background:#e8787a}.priority-medium{color:#c78b26;background:#fff7e7}.priority-medium::before{background:#e7b34e}.priority-low{color:#6e8c8a;background:#eef7f5}.priority-low::before{background:#80b6ad}.status-pill{background:#f2f3f7;color:#788397}.status-open{background:#fff6e6;color:#bc842b}.status-in_progress{background:#edf5ff;color:#4a83bd}.status-done{background:#eaf8f1;color:#49a77f}.due-date{font-size:9px;color:#8993a6}.due-date.overdue{color:#d8656b;font-weight:600}.row-actions{position:relative;display:flex;justify-content:flex-end;gap:3px}.row-action{border:0;background:transparent;color:#929bad;border-radius:5px;width:25px;height:25px;cursor:pointer;font-size:14px}.row-action:hover{background:#f0efff;color:#6259e8}.empty-state{padding:55px 20px 48px;text-align:center}.empty-state[hidden]{display:none}.empty-art{width:48px;height:48px;border-radius:15px;margin:0 auto 14px;background:linear-gradient(140deg,#f0efff,#f8f6ff);display:grid;place-items:center;color:#655be8;font-size:24px}.empty-state h3{font-size:14px;margin:0;color:#303c53}.empty-state p{font-size:10px;color:#98a1b0;margin:7px 0 14px}.text-button{background:none;border:0;color:#6259e8;font-family:inherit;font-size:10px;font-weight:600;cursor:pointer}.text-button span{font-size:14px;padding-left:3px}.page-footer{display:flex;align-items:center;gap:13px;margin:17px 3px 0;color:#a1a9b6;font-size:9px}.page-footer>span:first-child{font-size:8px;letter-spacing:1.1px;font-weight:700}.page-footer a{margin-left:auto;color:#8993a6;text-decoration:none}.page-footer a:hover{color:#6259e8}.page-footer i{display:inline-block;width:6px;height:6px;background:#54c997;border-radius:50%;margin-left:4px}.task-dialog{border:0;border-radius:14px;padding:0;width:min(460px,calc(100vw - 30px));box-shadow:0 25px 80px #111b3152;color:#243149}.task-dialog::backdrop{background:#111b3170;backdrop-filter:blur(3px)}.task-dialog form{padding:24px}.dialog-heading{display:flex;justify-content:space-between;align-items:flex-start;margin-bottom:22px}.dialog-heading h2{font-size:20px;letter-spacing:-.6px;margin:0}.dialog-heading .eyebrow{margin-bottom:6px}.icon-button{border:0;background:#f3f4f7;color:#788296;border-radius:7px;width:29px;height:29px;font-size:20px;cursor:pointer}.field-label{display:flex;align-items:center;gap:5px;font-size:10px;font-weight:650;color:#4a566b;margin:14px 0 6px}.field-label>span{color:#d76f73}.field-label small{margin-left:auto;color:#a0a8b6;font-size:9px;font-weight:400}.task-dialog input,.task-dialog textarea,.task-dialog select{width:100%;border:1px solid #e3e7ee;border-radius:7px;padding:10px 11px;color:#354158;background:#fff;font-family:inherit;font-size:11px;outline:0}.task-dialog input:focus,.task-dialog textarea:focus,.task-dialog select:focus{border-color:#8279ed;box-shadow:0 0 0 3px #8279ed20}.task-dialog textarea{resize:vertical;min-height:85px}.field-row{display:grid;grid-template-columns:1fr 1fr;gap:12px}.form-error{font-size:10px;color:#c5454b;background:#fff0f0;border-radius:6px;padding:9px 10px}.form-error[hidden]{display:none}.dialog-actions{display:flex;justify-content:flex-end;gap:8px;margin-top:22px}.dialog-actions button{min-width:80px}.toast{position:fixed;bottom:22px;left:50%;transform:translate(-50%,15px);opacity:0;pointer-events:none;background:#19253c;color:#fff;border-radius:8px;padding:11px 16px;font-size:11px;box-shadow:var(--shadow);transition:opacity .2s,transform .2s;z-index:5}.toast.show{opacity:1;transform:translate(-50%,0)}.toast.error{background:#a83c45}@keyframes appear{from{opacity:0;transform:translateY(5px)}to{opacity:1;transform:translateY(0)}}
@media(max-width:1050px){.sidebar{width:215px}.main-content{margin-left:215px;padding:0 30px 22px}.stats-grid{gap:9px}.list-head,.task-row{grid-template-columns:minmax(160px,1fr) 95px 110px 85px 30px;gap:8px}.list-head,.task-list{padding-left:15px;padding-right:15px}}
@media(max-width:760px){.sidebar{position:static;width:auto;height:auto;padding:14px 17px 10px;display:block}.brand{font-size:17px}.brand-mark{width:28px;height:28px}.workspace-label,.sidebar-bottom{display:none}.nav-list{display:flex;overflow:auto;gap:5px;margin-top:15px;padding-bottom:3px}.nav-item{white-space:nowrap;width:auto;padding:8px 10px;font-size:11px}.nav-count{display:none}.main-content{margin-left:0;padding:0 17px 20px}.topbar{height:55px}.page-heading{padding:28px 0 19px}.page-heading h1{font-size:25px}.subtitle{font-size:10px}.primary-button{min-height:36px;padding:0 12px;font-size:10px}.stats-grid{grid-template-columns:repeat(2,minmax(0,1fr));gap:8px;margin-bottom:14px}.stat-card{padding:12px}.stat-card>strong{font-size:21px;margin-top:7px}.panel-header{padding:15px 13px 12px;align-items:flex-start}.search-box{width:145px}.list-head{display:none}.task-list{padding:0 13px}.task-row{position:relative;grid-template-columns:minmax(0,1fr) auto;gap:7px;padding:13px 0;min-height:unset}.task-main{grid-column:1;grid-row:1/3}.priority{grid-column:2;grid-row:1;justify-self:end}.status-pill{grid-column:2;grid-row:2;justify-self:end}.due-date{grid-column:1;grid-row:3;padding-left:27px}.row-actions{position:absolute;right:0;bottom:12px}.page-footer{font-size:8px}.page-footer>span:nth-child(2){display:none}}
@media(prefers-reduced-motion:reduce){*,*::before,*::after{scroll-behavior:auto!important;animation-duration:.01ms!important;animation-iteration-count:1!important;transition-duration:.01ms!important}}


In [ ]:
%%writefile static/app.js
(() => {
  const dialog = document.querySelector("#task-dialog");
  const form = document.querySelector("#task-form");
  const list = document.querySelector("#task-list");
  const empty = document.querySelector("#empty-state");
  const toast = document.querySelector("#toast");
  const search = document.querySelector("#search");
  let activeFilter = "all";
  let editingId = null;
  let searchTimer = null;
  let toastTimer = null;

  const labels = { all: "All tasks", open: "To do", in_progress: "In progress", done: "Completed" };
  const dateFormatter = new Intl.DateTimeFormat(undefined, { month: "short", day: "numeric", year: "numeric" });

  function showToast(message, isError = false) {
    toast.textContent = message;
    toast.classList.toggle("error", isError);
    toast.classList.add("show");
    window.clearTimeout(toastTimer);
    toastTimer = window.setTimeout(() => toast.classList.remove("show"), 2800);
  }

  async function api(url, options = {}) {
    const response = await fetch(url, {
      ...options,
      headers: { Accept: "application/json", ...(options.body ? { "Content-Type": "application/json" } : {}), ...options.headers },
    });
    if (response.status === 204) return null;
    const data = await response.json().catch(() => ({}));
    if (!response.ok) throw new Error(data.error || "Something went wrong. Please try again.");
    return data;
  }

  function createElement(tag, className, text) {
    const element = document.createElement(tag);
    if (className) element.className = className;
    if (text !== undefined) element.textContent = text;
    return element;
  }

  function formatDate(value) {
    if (!value) return "No due date";
    const date = new Date(`${value}T00:00:00`);
    return dateFormatter.format(date);
  }

  function buildTask(task) {
    const row = createElement("article", `task-row${task.status === "done" ? " completed" : ""}`);
    const main = createElement("div", "task-main");
    const check = createElement("button", `task-check${task.status === "done" ? " is-done" : ""}`, task.status === "done" ? "✓" : "");
    check.type = "button";
    check.setAttribute("aria-label", task.status === "done" ? `Reopen ${task.title}` : `Complete ${task.title}`);
    check.addEventListener("click", () => changeStatus(task, task.status === "done" ? "open" : "done"));

    const text = createElement("div", "task-text");
    text.append(createElement("div", "task-title", task.title));
    if (task.description) text.append(createElement("div", "task-description", task.description));
    main.append(check, text);

    const priority = createElement("span", `priority priority-${task.priority}`, task.priority[0].toUpperCase() + task.priority.slice(1));
    const status = createElement("span", `status-pill status-${task.status}`, labels[task.status]);
    const due = createElement("time", "due-date", formatDate(task.due_date));
    if (task.due_date) {
      due.dateTime = task.due_date;
      if (task.status !== "done" && task.due_date < new Date().toISOString().slice(0, 10)) due.classList.add("overdue");
    }

    const actions = createElement("div", "row-actions");
    const edit = createElement("button", "row-action", "···");
    edit.type = "button";
    edit.setAttribute("aria-label", `Edit ${task.title}`);
    edit.title = "Edit task";
    edit.addEventListener("click", () => openEditor(task));
    const remove = createElement("button", "row-action", "×");
    remove.type = "button";
    remove.setAttribute("aria-label", `Delete ${task.title}`);
    remove.title = "Delete task";
    remove.addEventListener("click", () => deleteTask(task));
    actions.append(edit, remove);
    row.append(main, priority, status, due, actions);
    return row;
  }

  async function loadTasks() {
    list.setAttribute("aria-busy", "true");
    const params = new URLSearchParams();
    if (activeFilter !== "all") params.set("status", activeFilter);
    if (search.value.trim()) params.set("q", search.value.trim());
    try {
      const result = await api(`/api/tasks?${params.toString()}`);
      const fragment = document.createDocumentFragment();
      result.tasks.forEach((task) => fragment.append(buildTask(task)));
      list.replaceChildren(fragment);
      document.querySelector("#stat-total").textContent = result.stats.total;
      document.querySelector("#stat-open").textContent = result.stats.open;
      document.querySelector("#stat-progress").textContent = result.stats.in_progress;
      document.querySelector("#stat-done").textContent = result.stats.done;
      document.querySelector("#nav-total").textContent = result.stats.total;
      document.querySelector("#nav-open").textContent = result.stats.open;
      document.querySelector("#nav-progress").textContent = result.stats.in_progress;
      document.querySelector("#nav-done").textContent = result.stats.done;
      document.querySelector("#task-count").textContent = `${result.tasks.length} ${result.tasks.length === 1 ? "task" : "tasks"}${search.value.trim() ? " found" : " shown"}`;
      const noTasks = result.tasks.length === 0;
      empty.hidden = !noTasks;
      document.querySelector("#empty-title").textContent = search.value.trim() ? "No matches yet." : activeFilter === "done" ? "Nothing completed yet." : "A little room to think.";
      document.querySelector("#empty-copy").textContent = search.value.trim() ? "Try a different search, or clear the search field." : "Your task list is clear. Add a task to get started.";
      document.querySelector("#empty-add").hidden = Boolean(search.value.trim()) || activeFilter === "done";
    } catch (error) {
      list.replaceChildren();
      empty.hidden = false;
      document.querySelector("#empty-title").textContent = "Could not load your tasks.";
      document.querySelector("#empty-copy").textContent = error.message;
      document.querySelector("#empty-add").hidden = true;
    } finally {
      list.removeAttribute("aria-busy");
    }
  }

  async function changeStatus(task, status) {
    try {
      await api(`/api/tasks/${task.id}`, { method: "PATCH", body: JSON.stringify({ status }) });
      await loadTasks();
      showToast(status === "done" ? "Task completed." : "Task moved back to to do.");
    } catch (error) {
      showToast(error.message, true);
    }
  }

  async function deleteTask(task) {
    if (!window.confirm(`Delete “${task.title}”? This cannot be undone.`)) return;
    try {
      await api(`/api/tasks/${task.id}`, { method: "DELETE" });
      await loadTasks();
      showToast("Task deleted.");
    } catch (error) {
      showToast(error.message, true);
    }
  }

  function openEditor(task = null) {
    editingId = task ? task.id : null;
    form.reset();
    document.querySelector("#dialog-title").textContent = task ? "Edit task" : "Create a task";
    document.querySelector("#save-task").textContent = task ? "Save changes" : "Save task";
    document.querySelector("#title").value = task?.title || "";
    document.querySelector("#description").value = task?.description || "";
    document.querySelector("#priority").value = task?.priority || "medium";
    document.querySelector("#status").value = task?.status || "open";
    document.querySelector("#due-date").value = task?.due_date || "";
    document.querySelector("#form-error").hidden = true;
    dialog.showModal();
    document.querySelector("#title").focus();
  }

  document.querySelectorAll(".nav-item").forEach((button) => {
    button.addEventListener("click", () => {
      activeFilter = button.dataset.filter;
      document.querySelectorAll(".nav-item").forEach((item) => item.classList.toggle("active", item === button));
      document.querySelector("#view-name").textContent = labels[activeFilter];
      document.querySelector("#heading-title").firstChild.textContent = labels[activeFilter];
      loadTasks();
    });
  });

  document.querySelector("#add-task").addEventListener("click", () => openEditor());
  document.querySelector("#empty-add").addEventListener("click", () => openEditor());
  document.querySelector("#close-dialog").addEventListener("click", () => dialog.close());
  document.querySelector("#cancel-dialog").addEventListener("click", () => dialog.close());
  search.addEventListener("input", () => {
    window.clearTimeout(searchTimer);
    searchTimer = window.setTimeout(loadTasks, 180);
  });

  form.addEventListener("submit", async (event) => {
    event.preventDefault();
    const save = document.querySelector("#save-task");
    const errorBox = document.querySelector("#form-error");
    const data = Object.fromEntries(new FormData(form).entries());
    if (!data.due_date) data.due_date = null;
    save.disabled = true;
    errorBox.hidden = true;
    try {
      await api(editingId ? `/api/tasks/${editingId}` : "/api/tasks", {
        method: editingId ? "PATCH" : "POST",
        body: JSON.stringify(data),
      });
      dialog.close();
      await loadTasks();
      showToast(editingId ? "Changes saved." : "Task created.");
    } catch (error) {
      errorBox.textContent = error.message;
      errorBox.hidden = false;
    } finally {
      save.disabled = false;
    }
  });

  document.querySelector("#today-label").textContent = new Intl.DateTimeFormat(undefined, { weekday: "long", month: "long", day: "numeric" }).format(new Date()).toUpperCase();
  loadTasks();
})();


In [ ]:
%%writefile Dockerfile
FROM python:3.12-slim-bookworm

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1
WORKDIR /app
RUN useradd --create-home --uid 10001 appuser
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py ./
COPY templates ./templates
COPY static ./static
USER appuser
EXPOSE 5000
HEALTHCHECK --interval=30s --timeout=3s CMD python -c "import urllib.request; urllib.request.urlopen('http://127.0.0.1:5000/health')" || exit 1
CMD ["gunicorn", "--bind", "0.0.0.0:5000", "--workers", "2", "app:app"]


In [ ]:
%%writefile .dockerignore
.git
.venv
__pycache__
*.py[cod]
.env


In [ ]:
%%writefile .gitignore
.venv/
__pycache__/
*.py[cod]
.env
.pytest_cache/
*.db


In [ ]:
%%writefile sonar-project.properties
sonar.projectKey=YOUR_ORG_devsecops-security-gates
sonar.organization=YOUR_ORG
sonar.projectName=DevSecOps Security Gates Demo
sonar.sources=app.py
sonar.python.version=3.12
sonar.sourceEncoding=UTF-8


In [ ]:
%%writefile .github/workflows/security.yml
name: Security gates

on:
  push:
  pull_request:
  workflow_dispatch:

permissions:
  contents: read

jobs:
  secrets:
    name: Secret scan (Gitleaks)
    runs-on: ubuntu-latest
    permissions:
      contents: read
      pull-requests: write
    steps:
      - uses: actions/checkout@v6.0.2
        with:
          fetch-depth: 0
          persist-credentials: false
      - uses: gitleaks/gitleaks-action@v3.0.0
        env:
          GITHUB_TOKEN: ${{ github.token }}

  dependencies:
    name: Dependency scan (Trivy)
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v6.0.2
        with:
          persist-credentials: false
      - name: Scan files and dependencies
        uses: aquasecurity/trivy-action@v0.36.0
        with:
          scan-type: fs
          scan-ref: .
          scanners: vuln
          severity: HIGH,CRITICAL
          exit-code: '1'
          format: table
          github-pat: ${{ github.token }}

  sast:
    name: SAST and quality gate (SonarQube Cloud)
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v6.0.2
        with:
          fetch-depth: 0
          persist-credentials: false
      - name: Analyze and wait for quality gate
        uses: SonarSource/sonarqube-scan-action@v8.3.0
        with:
          args: >-
            -Dsonar.qualitygate.wait=true
            -Dsonar.qualitygate.timeout=300
        env:
          SONAR_TOKEN: ${{ secrets.SONAR_TOKEN }}

  container:
    name: Container scan (Trivy)
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v6.0.2
        with:
          persist-credentials: false
      - name: Build local image
        run: docker build --tag focusboard:${{ github.sha }} .
      - name: Scan image for HIGH/CRITICAL vulnerabilities
        uses: aquasecurity/trivy-action@v0.36.0
        with:
          image-ref: focusboard:${{ github.sha }}
          scan-type: image
          scanners: vuln
          severity: HIGH,CRITICAL
          exit-code: '1'
          format: table
          github-pat: ${{ github.token }}

  sbom:
    name: Generate SBOM
    runs-on: ubuntu-latest
    permissions:
      contents: read
    steps:
      - uses: actions/checkout@v6.0.2
        with:
          persist-credentials: false
      - name: Build image
        run: docker build --tag focusboard:${{ github.sha }} .
      - name: Generate CycloneDX SBOM
        uses: anchore/sbom-action@v0.24.0
        with:
          image: focusboard:${{ github.sha }}
          output-file: sbom-cyclonedx.json
          format: cyclonedx-json
          upload-artifact: false
          upload-release-assets: false
      - name: Upload SBOM artifact
        uses: actions/upload-artifact@v7.0.2
        with:
          name: sbom-cyclonedx
          path: sbom-cyclonedx.json
          if-no-files-found: error
          retention-days: 14


In [ ]:
%%writefile scripts/make-vulnerable.ps1
$ErrorActionPreference = 'Stop'

# This demo-only fixture is encoded so Gitleaks does not report a secret while
# scanning the clean default branch. Running it writes the intentional finding.
$encodedApp = 'aW1wb3J0IG9zCmltcG9ydCBzcWxpdGUzCmZyb20gZmxhc2sgaW1wb3J0IEZsYXNrLCBqc29uaWZ5LCByZXF1ZXN0CgphcHAgPSBGbGFzayhfX25hbWVfXykKZGVtb19rZXkgPSAiQUtJQTNIN0s5UDJSNlc0TjhYMVEiICAjIEZBS0UgQVdTLXNoYXBlZCB2YWx1ZSBmb3Igc2Nhbm5lciB0cmFpbmluZwpEQVRBQkFTRSA9IG9zLmVudmlyb24uZ2V0KCJEQVRBQkFTRV9QQVRIIiwgIi90bXAvdGFza3MuZGIiKQp3aXRoIHNxbGl0ZTMuY29ubmVjdChEQVRBQkFTRSkgYXMgY29ubjoKICAgIGNvbm4uZXhlY3V0ZSgiQ1JFQVRFIFRBQkxFIElGIE5PVCBFWElTVFMgdGFza3MgKGlkIElOVEVHRVIgUFJJTUFSWSBLRVksIHRpdGxlIFRFWFQgTk9UIE5VTEwsIG93bmVyIFRFWFQgTk9UIE5VTEwpIikKCkBhcHAuZ2V0KCIvdGFza3MiKQpkZWYgbGlzdF90YXNrcygpOgogICAgb3duZXIgPSByZXF1ZXN0LmFyZ3MuZ2V0KCJvd25lciIsICIiKQogICAgd2l0aCBzcWxpdGUzLmNvbm5lY3QoREFUQUJBU0UpIGFzIGNvbm46CiAgICAgICAgcm93cyA9IGNvbm4uZXhlY3V0ZSgiU0VMRUNUIGlkLCB0aXRsZSwgb3duZXIgRlJPTSB0YXNrcyBXSEVSRSBvd25lciA9ICclcyciICUgb3duZXIpCiAgICAgICAgcmV0dXJuIGpzb25pZnkocm93cy5mZXRjaGFsbCgpKQoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIGFwcC5ydW4oaG9zdD0iMC4wLjAuMCIsIHBvcnQ9NTAwMCkK'
[IO.File]::WriteAllBytes((Join-Path $PSScriptRoot '..\app.py'), [Convert]::FromBase64String($encodedApp))
Set-Content -Path (Join-Path $PSScriptRoot '..\requirements.txt') -Value @('Flask==2.0.1', 'Werkzeug==2.0.1', 'gunicorn==20.1.0')
Set-Content -Path (Join-Path $PSScriptRoot '..\Dockerfile') -Value @'
FROM python:3.9-slim-bullseye
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
EXPOSE 5000
CMD ["python", "app.py"]
'@
Write-Host 'Vulnerable demo state written. Review the diff, then commit and push this branch to trigger red gates.'


## Install and start

The next cell installs the pinned packages. Then start Flask. Open `http://127.0.0.1:5000` in a browser. Stop the server with the notebook interrupt control.

In [ ]:
%pip install -r requirements.txt

In [ ]:
from app import app
app.run(host="127.0.0.1", port=5000, debug=False)